# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR\u2072](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Access as object, not as a dict

print(f"{metadata.name}: {metadata.description}\n")print(f"Published: {metadata.datePublished}")print(f"Identifier: {metadata.identifier}")print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets and their fields and IDs.

In [ ]:
# List all record sets, their @ids, and their fields/columns
if hasattr(metadata, 'record_set') and metadata.record_set:
    print('Available record sets:')
    for recset in metadata.record_set:
        print(f"- RecordSet @id: {recset['@id']}")
        if 'field' in recset:
            for field in recset['field']:
                print(f"    - Field @id: {field['@id']} | name: {field.get('name', '')}")
            # check columns for this field
            for field in recset['field']:
                if 'column' in field:
                    for col in field['column']:
                        print(f"        - Column @id: {col['@id']} | name: {col.get('name', '')}")
else:
    print('No record sets found in the metadata.')

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Reference record set and field `@id`s found above.

In [ ]:
# Find all record set @ids
if hasattr(metadata, 'record_set') and metadata.record_set:
    record_sets_ids = [recset['@id'] for recset in metadata.record_set]
else:
    record_sets_ids = []

# Dictionary to keep DataFrames for each record set
dataframes = {}

for recset_id in record_sets_ids:
    print(f"Loading records from RecordSet @id: {recset_id}")
    try:
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded {len(df)} records. Columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Could not load data from record set {recset_id}: {e}\n")

if record_sets_ids:
    sample_rs = record_sets_ids[0]
    print(f"DataFrame columns for sample RecordSet '@id': {sample_rs}")
    print(dataframes[sample_rs].columns.tolist())
    display(dataframes[sample_rs].head())
else:
    print('No record sets to load.')

## 4. Exploratory Data Analysis (EDA)
Apply example data processing steps: filtering, normalizing a numeric field, and grouping.

In [ ]:
# Try to select the first numeric field (e.g., coverage, MW, peptide count)
import numpy as np
rs_id = None
num_field = None
group_field = None

# Heuristic: Find first record set with numeric columns
for k, df in dataframes.items():
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if numeric_candidates:
        rs_id = k
        num_field = numeric_candidates[0]
        # Try group field
        cat_candidates = [col for col in df.columns if df[col].dtype == object]
        group_field = cat_candidates[0] if cat_candidates else None
        break

if rs_id and num_field:
    threshold = dataframes[rs_id][num_field].quantile(0.75)  # Use Q3 as threshold
    filtered_df = dataframes[rs_id][dataframes[rs_id][num_field] > threshold]
    print(f"Filtered records from RecordSet {rs_id} with {num_field} > {threshold:.3f} ({len(filtered_df)} records):")
    display(filtered_df.head())

    # Normalize this field
    filtered_df[f"{num_field}_normalized"] = (filtered_df[num_field] - filtered_df[num_field].mean()) / filtered_df[num_field].std()
    print(f"Normalized {num_field} for filtered records:")
    display(filtered_df[[num_field, f"{num_field}_normalized"]].head())

    # Group by a categorical column, if one exists
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found for grouping.")
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Display a histogram of a numeric field and possibly a boxplot grouped by a categorical variable.

> **Note:** Visualization will be run based on the EDA-selected fields above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs_id and num_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[rs_id][num_field].dropna(), kde=True)
    plt.title(f"Distribution of '{num_field}' in RecordSet {rs_id}")
    plt.xlabel(num_field)
    plt.show()

    if group_field and group_field in dataframes[rs_id].columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=num_field, data=dataframes[rs_id])
        plt.title(f"Boxplot of '{num_field}' by '{group_field}'")
        plt.xticks(rotation=35)
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
This notebook demonstrated how to:
- Load dataset metadata and records using the Croissant schema and `mlcroissant`.
- Review available record sets and their fields using `@id` references.
- Extract records into DataFrames for further analysis.
- Perform filtering, normalization, and basic exploratory analysis on a numeric variable.
- Create visualizations to inspect distributions and grouped statistics.

Explore additional fields, groupings, and relationships in the record sets by further referencing their respective `@id`s!